In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
yelmo1=xr.open_dataset('../output/timeseries_from1D.nc')
yelmo_lgm=yelmo1.sel(time=slice(-19000, -16000))
n_best=2572

# Scores
scores = xr.open_dataset("../scoring/scores/scores_final.nc")

In [3]:
lgm_ens = []
lgm_ens_tot = []
idx_ens = []
for n_sim in yelmo_lgm.sim.values:
    sim = yelmo_lgm.sel(sim=n_sim)
    lgm = sim.V_sle.mean()
    pd = yelmo1.sel(time=0,sim=n_sim).V_sle
    lgm_ens.append(lgm-pd)
    lgm_ens_tot.append(lgm)
    idx_ens.append(n_sim)

lgm_ens = np.array(lgm_ens)
idx_ens = np.array(idx_ens)

LGM = xr.Dataset(
    data_vars={
        "dV_sle": (("sim"), lgm_ens),
        "V_sle": (("sim"), lgm_ens_tot),
    },
    coords={
        "sim": np.array(idx_ens),
    }
)

LGM.to_netcdf("../output/LGM_ens2.nc")

In [4]:
# Holocene minimal extension 
yelmo_hol=yelmo1.sel(time=slice(-7500, -2000))

hol_ens = []
hol_ens_tot = []
idx_ens = []
time_ens = []
for n_sim in yelmo_hol.sim.values:
    sim = yelmo_hol.sel(sim=n_sim)
    s = scores.sel(sim=n_sim).S.values

    hol = sim.V_sle.min()
    i = sim.V_sle.argmin().values
    t = sim.time[i].values

    pd = yelmo1.sel(time=0,sim=n_sim).V_sle
    hol_ens.append(hol-pd)
    hol_ens_tot.append(hol)
    idx_ens.append(n_sim)
    time_ens.append(t)

hol = xr.Dataset(
    data_vars={
        "dV_sle": (("sim"), hol_ens),
        "V_sle": (("sim"), hol_ens_tot),
        "time": (("sim"), time_ens),
    },
    coords={
        "sim": np.array(idx_ens),
    }
)
hol.to_netcdf("../output/HTM_ens2.nc")
